### Load packages

In [61]:
import pandas as pd
import os
import numpy as np
import zipfile
from typing import Union, Dict


### Define parameters and file info

In [62]:
# data_directory = r"C:\Users\hefla\Documents\Work\IPS\Industry Analysis\MCN"
data_directory = r"..\data"
output_directory = r"..\output"

# -----------------------------------------------------------------------
# DAF-to-DAF data
# -----------------------------------------------------------------------
daf2daf_file = r"MCN_DAF_to_DAF_grants_2023.csv"
daf2daf_path = os.path.join(data_directory, daf2daf_file)
print("DAF-to-DAF CSV file:", daf2daf_path)

# -----------------------------------------------------------------------
# All-DAF data
# -----------------------------------------------------------------------
alldaf_zipfile = r"MCN_all_DAF_grants_2023"
alldaf_path = os.path.join(data_directory, alldaf_zipfile + ".zip")
print("All-DAF zip file:", alldaf_path)

# -----------------------------------------------------------------------
# NTEE data
# -----------------------------------------------------------------------
ntee_directory = r"C:\Users\hefla\AppData\Local\Programs\Python\Python310\Lib\site-packages\irsx\CSV"
ntee_file = r"eo_bmf_extract_narrow"
ntee_path = os.path.join(ntee_directory, ntee_file + ".csv")
print("NTEE file:", ntee_path)

# -----------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------
full_output_file = r"AllDAF_grants.csv"
full_output_path = os.path.join(output_directory, full_output_file)
print("Full output file:", full_output_path)
nod2d_output_file = r"AllDAF_grants_no_DAF2DAF.csv"
nod2doutput_path = os.path.join(output_directory, nod2d_output_file)
print("NoD2D output file:", nod2doutput_path)
grantor_summary_file = r"DAF_grants_summary_by_grantor_NTEE.csv"
grantor_summary_path = os.path.join(output_directory, grantor_summary_file)
print("Grantor Summary file:", grantor_summary_path)
recipient_summary_file = r"DAF_grants_summary_by_recipient_NTEE.csv"
recipient_summary_path = os.path.join(output_directory, recipient_summary_file)
print("Grantor Summary file:", recipient_summary_path)


DAF-to-DAF CSV file: ..\data\MCN_DAF_to_DAF_grants_2023.csv
All-DAF zip file: ..\data\MCN_all_DAF_grants_2023.zip
NTEE file: C:\Users\hefla\AppData\Local\Programs\Python\Python310\Lib\site-packages\irsx\CSV\eo_bmf_extract_narrow.csv
Full output file: ..\output\AllDAF_grants.csv
NoD2D output file: ..\output\AllDAF_grants_no_DAF2DAF.csv
Grantor Summary file: ..\output\DAF_grants_summary_by_grantor_NTEE.csv
Grantor Summary file: ..\output\DAF_grants_summary_by_recipient_NTEE.csv


In [63]:
# Function to read in zipped files into a dictionary of dataframes

def load_csv_from_zip(src: str) -> Union[pd.DataFrame, Dict[str, pd.DataFrame]]:

    with zipfile.ZipFile(src, "r") as archive:
        # keep only CSV files
        csv_files = [f for f in archive.namelist() if f.lower().endswith(".csv")]

        if len(csv_files) == 0:
            raise ValueError("No CSV files found in the zip archive.")

        # If there is only one CSV file, return a dataframe
        if len(csv_files) == 1:
            with archive.open(csv_files[0]) as file:
                return pd.read_csv(file)

        # Otherwise return a dictionary of dataframes
        dfs = {}
        for filename in csv_files:
            with archive.open(filename) as file:
                dfs[filename] = pd.read_csv(file)

        return dfs

### Load in data

In [64]:
# Load DAF-to-DAF data

daf2daf_raw_df = pd.read_csv(daf2daf_path)

print(daf2daf_raw_df.info())

daf2daf_raw_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Grantor EIN             1447 non-null   int64 
 1   Grantor Name            1447 non-null   object
 2   Recipient EIN           1447 non-null   int64 
 3   Recipient Name          1447 non-null   object
 4   Cash Grant Amount       1447 non-null   object
 5   Grantor/Recipient Type  1447 non-null   object
dtypes: int64(2), object(4)
memory usage: 68.0+ KB
None


,Grantor EIN,Grantor Name,Recipient EIN,Recipient Name,Cash Grant Amount,Grantor/Recipient Type
0,10782573,ADVISORS CHARITABLE GIFT FUND INC,110303001,FIDELITY INVESTMENTS CHARITABLE GIFT FUND,"$204,922",Nat-to-Nat
1,10782573,ADVISORS CHARITABLE GIFT FUND INC,352129262,RENAISSANCE CHARITABLE FOUNDATION INC,"$455,168",Nat-to-Nat
2,10782573,ADVISORS CHARITABLE GIFT FUND INC,527082731,MORGAN STANLEY GLOBAL IMPACT FUNDING TRUST INC,"$220,194",Nat-to-Nat
3,10782573,ADVISORS CHARITABLE GIFT FUND INC,593652538,RAYMOND JAMES CHARITABLE ENDOWMENT FUND,"$254,975",Nat-to-Nat
4,46649138,FIDUCIARY CHARITABLE FOUNDATION DBA FIDUCIARY ...,110303001,FIDELITY INVESTMENTS CHARITABLE GIFT FUND,"$10,000",Nat-to-Nat


In [65]:
# Load All-DAF data

alldaf_raw_df = load_csv_from_zip(alldaf_path)

alldaf_raw_df.head()

C:\Users\hefla\AppData\Local\Temp\ipykernel_23248\685997372.py:15: DtypeWarning: Columns (13,14,17,18,19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)


,Object ID,EIN,Organization Name,Year of Tax Period End Date,RcpntBsnssNm_BsnssNmLn1Txt,USAddrss_AddrssLn1Txt,USAddrss_CtyNm,USAddrss_SttAbbrvtnCd,USAddrss_ZIPCd,RcpntTbl_RcpntEIN,...,RcpntTbl_VltnMthdUsdDsc,RcpntTbl_NnCshAssstncDsc,USAddrss_AddrssLn2Txt,RcpntTbl_NnCshAssstncAmt,RcpntBsnssNm_BsnssNmLn2Txt,FrgnAddrss_AddrssLn1Txt,FrgnAddrss_CtyNm,FrgnAddrss_CntryCd,FrgnAddrss_PrvncOrSttNm,FrgnAddrss_FrgnPstlCd
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,317 MAIN ST COMMUNITY MUSIC CENTER INC,317 MAIN STREET,YARMOUTH,ME,4096.0,201424631.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,350VERMONT,239 SOUTH UNION STREET SUITE 3,BURLINGTON,VT,5401.0,463647561.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,3I HOUSING OF MAINE,4 UNION PARK SUITE 3I,TOPSHAM,ME,4086.0,852568325.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A CLIMATE TO THRIVE,PO BOX 75,MOUNT DESERT,ME,4660.0,810836597.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A COMPANY OF GIRLS,PO BOX 7527,PORTLAND,ME,4112.0,50631726.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
# Load in NTEE data

ntee_raw_df = pd.read_csv(ntee_path)

print(ntee_raw_df.info())

ntee_raw_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2243322 entries, 0 to 2243321
Data columns (total 9 columns):
 #   Column               Dtype  
---  ------               -----  
 0   EIN                  float64
 1   ORG_NAME_CURRENT     object 
 2   BMF_SUBSECTION_CODE  int64  
 3   F990_ORG_ADDR_STATE  object 
 4   NCCS_LEVEL_1         object 
 5   NTEE_IRS             object 
 6   NTEE_NCCS            object 
 7   NTEEV2               object 
 8   NTEE_COMBINED        object 
dtypes: float64(1), int64(1), object(7)
memory usage: 154.0+ MB
None


,EIN,ORG_NAME_CURRENT,BMF_SUBSECTION_CODE,F990_ORG_ADDR_STATE,NCCS_LEVEL_1,NTEE_IRS,NTEE_NCCS,NTEEV2,NTEE_COMBINED
0,1.0,VOLUNTEERS OF AMERICA INC,3,KY,501C3 CHARITY,B43,B43,UNI-B43-RG,B43
1,3154.0,OAKLEAF FOREST TENANT MANAGEMENT,3,VA,501C3 CHARITY,C36,C36,ENV-C36-RG,C36
2,4101.0,SOUTH LAFOURCHE QUARTERBACK CLUB,3,LA,501C3 PRIVATE FOUNDATION,N65,N65,HMS-N65-RG,N65
3,19818.0,PALMER SECOND BAPTIST CHURCH,3,MA,501C3 CHARITY,X21,X21,REL-X21-RG,X21
4,29215.0,ST GEORGE CATHEDRAL,3,MA,501C3 CHARITY,X99,X99,REL-X99-RG,X99


### Clean and format columns

In [67]:
# Clean & format columns in DAF2DAF data

daf2daf_clean_df = daf2daf_raw_df.copy()

# Convert integer columns to integer format
int_cols = ["Grantor EIN", "Recipient EIN"]
daf2daf_clean_df[int_cols] = daf2daf_clean_df[int_cols].astype("Int64")

# Clean up dollar formatting in CSV file
daf2daf_clean_df["Cash Grant Amount"] = (
    daf2daf_clean_df["Cash Grant Amount"]
        .str.replace(r"[$,]", "", regex=True)
        .astype(float)
)
# Convert to float
daf2daf_clean_df["Cash Grant Amount"] = daf2daf_clean_df["Cash Grant Amount"].astype(float)

daf2daf_clean_df.head()

,Grantor EIN,Grantor Name,Recipient EIN,Recipient Name,Cash Grant Amount,Grantor/Recipient Type
0,10782573,ADVISORS CHARITABLE GIFT FUND INC,110303001,FIDELITY INVESTMENTS CHARITABLE GIFT FUND,204922.0,Nat-to-Nat
1,10782573,ADVISORS CHARITABLE GIFT FUND INC,352129262,RENAISSANCE CHARITABLE FOUNDATION INC,455168.0,Nat-to-Nat
2,10782573,ADVISORS CHARITABLE GIFT FUND INC,527082731,MORGAN STANLEY GLOBAL IMPACT FUNDING TRUST INC,220194.0,Nat-to-Nat
3,10782573,ADVISORS CHARITABLE GIFT FUND INC,593652538,RAYMOND JAMES CHARITABLE ENDOWMENT FUND,254975.0,Nat-to-Nat
4,46649138,FIDUCIARY CHARITABLE FOUNDATION DBA FIDUCIARY ...,110303001,FIDELITY INVESTMENTS CHARITABLE GIFT FUND,10000.0,Nat-to-Nat


In [68]:
# Clean & format columns in All-DAF data

alldaf_clean_df = alldaf_raw_df.copy()

# Convert integer columns to integer format
int_cols = ["Object ID", "EIN", "Year of Tax Period End Date", "RcpntTbl_RcpntEIN", "USAddrss_ZIPCd"]
alldaf_raw_df[int_cols] = alldaf_raw_df[int_cols].astype("Int64")

# Clean up dollar formatting in CSV file
dollar_cols = ["RcpntTbl_CshGrntAmt", "RcpntTbl_NnCshAssstncAmt"]
# Convert to float
alldaf_clean_df[dollar_cols] = alldaf_clean_df[dollar_cols].astype(float)

alldaf_clean_df.head()

,Object ID,EIN,Organization Name,Year of Tax Period End Date,RcpntBsnssNm_BsnssNmLn1Txt,USAddrss_AddrssLn1Txt,USAddrss_CtyNm,USAddrss_SttAbbrvtnCd,USAddrss_ZIPCd,RcpntTbl_RcpntEIN,...,RcpntTbl_VltnMthdUsdDsc,RcpntTbl_NnCshAssstncDsc,USAddrss_AddrssLn2Txt,RcpntTbl_NnCshAssstncAmt,RcpntBsnssNm_BsnssNmLn2Txt,FrgnAddrss_AddrssLn1Txt,FrgnAddrss_CtyNm,FrgnAddrss_CntryCd,FrgnAddrss_PrvncOrSttNm,FrgnAddrss_FrgnPstlCd
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,317 MAIN ST COMMUNITY MUSIC CENTER INC,317 MAIN STREET,YARMOUTH,ME,4096.0,201424631.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,350VERMONT,239 SOUTH UNION STREET SUITE 3,BURLINGTON,VT,5401.0,463647561.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,3I HOUSING OF MAINE,4 UNION PARK SUITE 3I,TOPSHAM,ME,4086.0,852568325.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A CLIMATE TO THRIVE,PO BOX 75,MOUNT DESERT,ME,4660.0,810836597.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A COMPANY OF GIRLS,PO BOX 7527,PORTLAND,ME,4112.0,50631726.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [69]:
# QC revenue

print(alldaf_clean_df["RcpntTbl_CshGrntAmt"].sum())

62004522644.0


### Clean and format columns in NTEE data

In [70]:
# Clean & format columns in NTEE data

ntee_clean_df = ntee_raw_df.copy()

# Convert integer columns to integer format
int_cols = ["EIN", "BMF_SUBSECTION_CODE"]
ntee_clean_df[int_cols] = ntee_clean_df[int_cols].astype("Int64")

# Drop unnecessary columns
cols_to_drop = ["F990_ORG_ADDR_STATE", "NCCS_LEVEL_1", "NTEE_IRS", "NTEE_NCCS", "NTEEV2"]
ntee_clean_df = ntee_clean_df.drop(columns=cols_to_drop)

# Uppercase NTEE_COMBINED
ntee_clean_df["NTEE_COMBINED"] = ntee_clean_df["NTEE_COMBINED"].str.upper()

# Create one-digit NTEE code columm
ntee_clean_df["NTEE_FIRST_DIGIT"] = ntee_clean_df["NTEE_COMBINED"].str[0]

# Identify invalid first digits (not A-Z or null/NaN) and replace with "0"
ntee_clean_df["NTEE_FIRST_DIGIT"] = ntee_clean_df["NTEE_FIRST_DIGIT"].mask(
    ntee_clean_df["NTEE_FIRST_DIGIT"].isna() | ~ntee_clean_df["NTEE_FIRST_DIGIT"].str.match(r"^[A-Z]$", na=False),
    "0"
)

ntee_clean_df.head()

,EIN,ORG_NAME_CURRENT,BMF_SUBSECTION_CODE,NTEE_COMBINED,NTEE_FIRST_DIGIT
0,1,VOLUNTEERS OF AMERICA INC,3,B43,B
1,3154,OAKLEAF FOREST TENANT MANAGEMENT,3,C36,C
2,4101,SOUTH LAFOURCHE QUARTERBACK CLUB,3,N65,N
3,19818,PALMER SECOND BAPTIST CHURCH,3,X21,X
4,29215,ST GEORGE CATHEDRAL,3,X99,X


In [71]:
# Find records where NTEE_COMBINED is null or 0

# Rows where NTEE_FIRST_DIGIT is NaN or not A-Z
invalid_ntee_rows = ntee_clean_df[
    ntee_clean_df["NTEE_FIRST_DIGIT"].isna() |        # null/NaN
    ~ntee_clean_df["NTEE_FIRST_DIGIT"].str.match(r"^[A-Z]$", na=False)  # not a single uppercase letter
]

# print(invalid_ntee_rows.value_counts())
# Show the results
# print(invalid_ntee_rows)
print("Number of invalid NTEE codes:", len(invalid_ntee_rows))

Number of invalid NTEE codes: 53654


In [72]:
# Dedupe the NTEE file

# Remove rows that are fully identical
ntee_dedupe_fullrows_df = ntee_clean_df.drop_duplicates(keep="first")

# Check how many full-row duplicates were removed
num_removed = len(ntee_clean_df) - len(ntee_dedupe_fullrows_df)
print(f"Removed {num_removed} fully identical duplicate rows.")

Removed 2361 fully identical duplicate rows.


In [73]:
# This will return all rows where EIN is duplicated
duplicates_df = ntee_dedupe_fullrows_df[ntee_dedupe_fullrows_df["EIN"].duplicated(keep=False)]

print(f"Total duplicate rows: {len(duplicates_df)}")
duplicates_df.head(10)

Total duplicate rows: 452


,EIN,ORG_NAME_CURRENT,BMF_SUBSECTION_CODE,NTEE_COMBINED,NTEE_FIRST_DIGIT
2239,10458555,SCARBOROUGH FISH & GAME ASSOCIATION INC,3,B90,B
2241,10458555,SCARBOROUGH FISH & GAME ASSOCIATION INC,3,N50,N
2309,10463535,SAFARI CLUB INTERNATIONAL FOUNDATION,3,D30,D
2311,10463535,SAFARI CLUB INTERNATIONAL FOUNDATION,3,N0160,N
20282,357000000,PRAIRIETON VOLUNTEER FIREMENS ASSOCIATION INC,4,NaN,0
20284,357000000,PRAIRIETON VOLUNTEER FIREMENS ASSOCIATION INC,4,M24,M
20286,358000000,INTERNATIONAL ASSOCIATION OF MACHINISTS & AERO...,5,NaN,0
20288,358000000,INTERNATIONAL ASSOCIATION OF MACHINISTS & AERO...,5,J40,J
20770,30221525,LAMOILLE ECONOMIC DEVELOPMENT CORPORATION INC,6,S30,S
20772,30221525,LAMOILLE ECONOMIC DEVELOPMENT CORPORATION INC,6,S41,S


In [74]:
# Function to select the row we want for each EIN
def pick_ntee_row(group):
    # Rows where NTEE_FIRST_DIGIT is not "0"
    non_zero_rows = group[group["NTEE_FIRST_DIGIT"] != "0"]
    
    if len(non_zero_rows) > 0:
        return non_zero_rows.iloc[0]   # first valid record
    else:
        return group.iloc[0]           # first invalid record if none valid

In [75]:
# Dedupe remaining rows by choosing first valid, or first invalid if no valid record exists

# Only keep EIN duplicates to speed up
duplicates_df = ntee_dedupe_fullrows_df[ntee_dedupe_fullrows_df.duplicated(subset="EIN", keep=False)]

# Apply your function only to these duplicates
deduped_duplicates = duplicates_df.groupby("EIN", group_keys=False).apply(pick_ntee_row)

# Keep the rows that were unique to start with
unique_rows = ntee_dedupe_fullrows_df.drop_duplicates(subset="EIN", keep=False)

# Combine back
ntee_dedupe_df = pd.concat([unique_rows, deduped_duplicates], ignore_index=True)

print(f"Original records: {len(ntee_dedupe_fullrows_df)}")
print(f"After deduping by EIN: {len(ntee_dedupe_df)}")

C:\Users\hefla\AppData\Local\Temp\ipykernel_23248\308749765.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  deduped_duplicates = duplicates_df.groupby("EIN", group_keys=False).apply(pick_ntee_row)


Original records: 2240961
After deduping by EIN: 2240733


In [76]:
# Check duplicate rows again
duplicates_df = ntee_dedupe_df[ntee_dedupe_df["EIN"].duplicated(keep=False)]

print(f"Total duplicate rows: {len(duplicates_df)}")
# duplicates_df.head(10)

Total duplicate rows: 0


### Add NTEE codes to All-DAF grants file

In [77]:
# Add NTEE codes for grantors

print("Before merging:", len(alldaf_clean_df))
print(alldaf_clean_df["RcpntTbl_CshGrntAmt"].sum())

alldaf_grantor_ntee_df = (
    alldaf_clean_df
    .merge(
        ntee_dedupe_df[["EIN", "NTEE_COMBINED", "NTEE_FIRST_DIGIT"]],
        left_on=["EIN"],
        right_on=["EIN"],
        how="left",
        indicator=True
    )
)

# Drop helper columns
alldaf_grantor_ntee_df = alldaf_grantor_ntee_df.drop(columns=["_merge"])

# Rename columns
alldaf_grantor_ntee_df = alldaf_grantor_ntee_df.rename(
    columns={
        "NTEE_COMBINED": "GRANTOR_NTEE_COMBINED",
        "NTEE_FIRST_DIGIT": "GRANTOR_NTEE_FIRST_DIGIT"
    }
)

print("After merging:", len(alldaf_grantor_ntee_df))
print(alldaf_grantor_ntee_df["RcpntTbl_CshGrntAmt"].sum())

alldaf_grantor_ntee_df.head()


Before merging: 518889
62004522644.0
After merging: 518889
62004522644.0


,Object ID,EIN,Organization Name,Year of Tax Period End Date,RcpntBsnssNm_BsnssNmLn1Txt,USAddrss_AddrssLn1Txt,USAddrss_CtyNm,USAddrss_SttAbbrvtnCd,USAddrss_ZIPCd,RcpntTbl_RcpntEIN,...,USAddrss_AddrssLn2Txt,RcpntTbl_NnCshAssstncAmt,RcpntBsnssNm_BsnssNmLn2Txt,FrgnAddrss_AddrssLn1Txt,FrgnAddrss_CtyNm,FrgnAddrss_CntryCd,FrgnAddrss_PrvncOrSttNm,FrgnAddrss_FrgnPstlCd,GRANTOR_NTEE_COMBINED,GRANTOR_NTEE_FIRST_DIGIT
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,317 MAIN ST COMMUNITY MUSIC CENTER INC,317 MAIN STREET,YARMOUTH,ME,4096.0,201424631.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,T31,T
1,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,350VERMONT,239 SOUTH UNION STREET SUITE 3,BURLINGTON,VT,5401.0,463647561.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,T31,T
2,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,3I HOUSING OF MAINE,4 UNION PARK SUITE 3I,TOPSHAM,ME,4086.0,852568325.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,T31,T
3,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A CLIMATE TO THRIVE,PO BOX 75,MOUNT DESERT,ME,4660.0,810836597.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,T31,T
4,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A COMPANY OF GIRLS,PO BOX 7527,PORTLAND,ME,4112.0,50631726.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,T31,T


In [78]:
# Add NTEE codes for recipients

print("Before merging:", len(alldaf_grantor_ntee_df))

# Select only the relevant columns from ntee_dedupe_df
recipient_ntee_df = ntee_dedupe_df[["EIN", "NTEE_COMBINED", "NTEE_FIRST_DIGIT"]].rename(
    columns={
        "EIN": "RECIPIENT_EIN",
        "NTEE_COMBINED": "RECIPIENT_NTEE_COMBINED",
        "NTEE_FIRST_DIGIT": "RECIPIENT_NTEE_FIRST_DIGIT"
    }
)

alldaf_allntee_df = alldaf_grantor_ntee_df.merge(
    recipient_ntee_df,
    left_on="RcpntTbl_RcpntEIN",
    right_on="RECIPIENT_EIN",
    how="left"
)

# Drop the merge helper columns
alldaf_allntee_df = alldaf_allntee_df.drop(columns=["RECIPIENT_EIN"])
alldaf_allntee_df = alldaf_allntee_df.drop(columns=["_merge"], errors="ignore")

print("After merging:", len(alldaf_allntee_df))
print(alldaf_allntee_df["RcpntTbl_CshGrntAmt"].sum())

alldaf_allntee_df.head()


Before merging: 518889
After merging: 518889
62004522644.0


,Object ID,EIN,Organization Name,Year of Tax Period End Date,RcpntBsnssNm_BsnssNmLn1Txt,USAddrss_AddrssLn1Txt,USAddrss_CtyNm,USAddrss_SttAbbrvtnCd,USAddrss_ZIPCd,RcpntTbl_RcpntEIN,...,RcpntBsnssNm_BsnssNmLn2Txt,FrgnAddrss_AddrssLn1Txt,FrgnAddrss_CtyNm,FrgnAddrss_CntryCd,FrgnAddrss_PrvncOrSttNm,FrgnAddrss_FrgnPstlCd,GRANTOR_NTEE_COMBINED,GRANTOR_NTEE_FIRST_DIGIT,RECIPIENT_NTEE_COMBINED,RECIPIENT_NTEE_FIRST_DIGIT
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,317 MAIN ST COMMUNITY MUSIC CENTER INC,317 MAIN STREET,YARMOUTH,ME,4096.0,201424631.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,A6E,A
1,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,350VERMONT,239 SOUTH UNION STREET SUITE 3,BURLINGTON,VT,5401.0,463647561.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,C60,C
2,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,3I HOUSING OF MAINE,4 UNION PARK SUITE 3I,TOPSHAM,ME,4086.0,852568325.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,L24,L
3,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A CLIMATE TO THRIVE,PO BOX 75,MOUNT DESERT,ME,4660.0,810836597.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,C35,C
4,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A COMPANY OF GIRLS,PO BOX 7527,PORTLAND,ME,4112.0,50631726.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,A65,A


### Remove DAF-to-DAF grants from All-DAF grants

In [79]:
# Find DAF2DAF duplicates in All-DAF grants list

# Find the matching rows in the 2 dataframes
matched = alldaf_allntee_df.merge(
    daf2daf_clean_df[["Grantor EIN", "Recipient EIN"]],
    left_on=["EIN", "RcpntTbl_RcpntEIN"],
    right_on=["Grantor EIN", "Recipient EIN"],
    how="inner"
)

# Identify duplicates in those rows
duplicates = matched[
    matched.duplicated(
        subset=["EIN", "RcpntTbl_RcpntEIN"],
        keep=False
    )
]

duplicates = duplicates.sort_values(["EIN", "RcpntTbl_RcpntEIN"])
print(duplicates[["EIN", "RcpntTbl_RcpntEIN", "RcpntTbl_CshGrntAmt"]])

            EIN  RcpntTbl_RcpntEIN  RcpntTbl_CshGrntAmt
18     46649138        510198509.0            1000000.0
19     46649138        510198509.0             761146.0
20     61676688        341747398.0           76391938.0
25     61676688        341747398.0           86191082.0
26     61676688        516506426.0              14181.0
...         ...                ...                  ...
1445  841260437        110303001.0               5822.0
1484  943136771        943136771.0              40850.0
1485  943136771        943136771.0             241000.0
1486  946070996        943136771.0              37500.0
1492  946070996        943136771.0              40000.0

[137 rows x 3 columns]


In [80]:
# Remove DAF2DAF grants from All-DAF grants list

print("Before filtering:", len(alldaf_allntee_df))
print("Records in DAF2DAF list:", len(daf2daf_clean_df))

alldaf_filtered_df = (
    alldaf_allntee_df
    .merge(
        daf2daf_clean_df[["Grantor EIN", "Recipient EIN"]],
        left_on=["EIN", "RcpntTbl_RcpntEIN"],
        right_on=["Grantor EIN", "Recipient EIN"],
        how="left",
        indicator=True
    )
)

# Keep only rows that did NOT match
alldaf_filtered_df = alldaf_filtered_df[alldaf_filtered_df["_merge"] == "left_only"]

# Drop helper columns
alldaf_filtered_df = alldaf_filtered_df.drop(columns=["Grantor EIN", "Recipient EIN", "_merge"])

print("After filtering:", len(alldaf_filtered_df))
print(alldaf_filtered_df["RcpntTbl_CshGrntAmt"].sum())

alldaf_filtered_df.head()

Before filtering: 518889
Records in DAF2DAF list: 1447
After filtering: 517353
57017667108.0


,Object ID,EIN,Organization Name,Year of Tax Period End Date,RcpntBsnssNm_BsnssNmLn1Txt,USAddrss_AddrssLn1Txt,USAddrss_CtyNm,USAddrss_SttAbbrvtnCd,USAddrss_ZIPCd,RcpntTbl_RcpntEIN,...,RcpntBsnssNm_BsnssNmLn2Txt,FrgnAddrss_AddrssLn1Txt,FrgnAddrss_CtyNm,FrgnAddrss_CntryCd,FrgnAddrss_PrvncOrSttNm,FrgnAddrss_FrgnPstlCd,GRANTOR_NTEE_COMBINED,GRANTOR_NTEE_FIRST_DIGIT,RECIPIENT_NTEE_COMBINED,RECIPIENT_NTEE_FIRST_DIGIT
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,317 MAIN ST COMMUNITY MUSIC CENTER INC,317 MAIN STREET,YARMOUTH,ME,4096.0,201424631.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,A6E,A
1,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,350VERMONT,239 SOUTH UNION STREET SUITE 3,BURLINGTON,VT,5401.0,463647561.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,C60,C
2,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,3I HOUSING OF MAINE,4 UNION PARK SUITE 3I,TOPSHAM,ME,4086.0,852568325.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,L24,L
3,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A CLIMATE TO THRIVE,PO BOX 75,MOUNT DESERT,ME,4660.0,810836597.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,C35,C
4,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,A COMPANY OF GIRLS,PO BOX 7527,PORTLAND,ME,4112.0,50631726.0,...,NaN,NaN,NaN,NaN,NaN,NaN,T31,T,A65,A


### Create summaries by NTEE code

In [81]:
# Check for missing values in RECIPIENT_NTEE_FIRST_DIGIT
missing_count = alldaf_allntee_df["GRANTOR_NTEE_FIRST_DIGIT"].isna().sum()
print(f"Missing GRANTOR_NTEE_FIRST_DIGIT in alldaf_allntee_df: {missing_count}")

missing_filtered_count = alldaf_filtered_df["GRANTOR_NTEE_FIRST_DIGIT"].isna().sum()
print(f"Missing GRANTOR_NTEE_FIRST_DIGIT in alldaf_filtered_df: {missing_filtered_count}")

# Optionally, see which values are invalid (not a letter A-Z or 0)
invalid_values = alldaf_allntee_df.loc[
    ~alldaf_allntee_df["GRANTOR_NTEE_FIRST_DIGIT"].astype(str).str.match("^[A-Z0-9]$"),
    "GRANTOR_NTEE_FIRST_DIGIT"
].unique()
print(f"Other invalid GRANTOR_NTEE_FIRST_DIGIT values: {invalid_values}")

Missing GRANTOR_NTEE_FIRST_DIGIT in alldaf_allntee_df: 6
Missing GRANTOR_NTEE_FIRST_DIGIT in alldaf_filtered_df: 6
Other invalid GRANTOR_NTEE_FIRST_DIGIT values: [nan]


In [82]:
# Check for missing values in RECIPIENT_NTEE_FIRST_DIGIT
missing_count = alldaf_allntee_df["RECIPIENT_NTEE_FIRST_DIGIT"].isna().sum()
print(f"Missing RECIPIENT_NTEE_FIRST_DIGIT in alldaf_allntee_df: {missing_count}")

missing_filtered_count = alldaf_filtered_df["RECIPIENT_NTEE_FIRST_DIGIT"].isna().sum()
print(f"Missing RECIPIENT_NTEE_FIRST_DIGIT in alldaf_filtered_df: {missing_filtered_count}")

# Optionally, see which values are invalid (not a letter A-Z or 0)
invalid_values = alldaf_allntee_df.loc[
    ~alldaf_allntee_df["RECIPIENT_NTEE_FIRST_DIGIT"].astype(str).str.match("^[A-Z0-9]$"),
    "RECIPIENT_NTEE_FIRST_DIGIT"
].unique()
print(f"Other invalid RECIPIENT_NTEE_FIRST_DIGIT values: {invalid_values}")

Missing RECIPIENT_NTEE_FIRST_DIGIT in alldaf_allntee_df: 60588
Missing RECIPIENT_NTEE_FIRST_DIGIT in alldaf_filtered_df: 60588
Other invalid RECIPIENT_NTEE_FIRST_DIGIT values: [nan]


In [83]:
# GRANTOR Summary for the full/clean dataframe

# Full/clean summary
summary_original = (
    alldaf_allntee_df
    .groupby("GRANTOR_NTEE_FIRST_DIGIT", as_index=False, dropna=False)  # keep NaNs as groups
    .agg(
        Total_Cash_Grants_Original=("RcpntTbl_CshGrntAmt", "sum"),
        Count_Original=("RcpntTbl_CshGrntAmt", "count")
    )
)

# Filtered summary
summary_no_DAF2DAF = (
    alldaf_filtered_df
    .groupby("GRANTOR_NTEE_FIRST_DIGIT", as_index=False, dropna=False)
    .agg(
        Total_Cash_Grants_No_DAF2DAF=("RcpntTbl_CshGrntAmt", "sum"),
        Count_No_DAF2DAF=("RcpntTbl_CshGrntAmt", "count")
    )
)

# Merge with outer join
grantor_summary_combined = summary_original.merge(
    summary_no_DAF2DAF,
    on="GRANTOR_NTEE_FIRST_DIGIT",
    how="outer"
)

# Difference columns
grantor_summary_combined["DAF2DAF_Grants"] = (
    grantor_summary_combined["Total_Cash_Grants_Original"] - 
    grantor_summary_combined["Total_Cash_Grants_No_DAF2DAF"]
)
grantor_summary_combined["Count_DAF2DAF_Grants"] = (
    grantor_summary_combined["Count_Original"] - 
    grantor_summary_combined["Count_No_DAF2DAF"]
)

# Optional: sort so NaNs come first
grantor_summary_combined = grantor_summary_combined.sort_values(
    by="GRANTOR_NTEE_FIRST_DIGIT", na_position="first"
)

grantor_summary_combined.head(30)

# print(grantor_summary_combined["Total_Cash_Grants_No_DAF2DAF"].sum())

,GRANTOR_NTEE_FIRST_DIGIT,Total_Cash_Grants_Original,Count_Original,Total_Cash_Grants_No_DAF2DAF,Count_No_DAF2DAF,DAF2DAF_Grants,Count_DAF2DAF_Grants
26,NaN,8.681300e+04,6,8.681300e+04,6,0.000000e+00,0
0,0,1.134554e+07,29,1.134554e+07,29,0.000000e+00,0
1,A,3.217522e+07,487,3.217522e+07,487,0.000000e+00,0
2,B,3.557361e+09,9147,3.555562e+09,9134,1.799035e+06,13
3,C,1.201998e+08,262,1.201998e+08,262,0.000000e+00,0
4,D,9.489883e+07,70,9.489883e+07,70,0.000000e+00,0
5,E,1.135029e+09,1034,1.135029e+09,1034,0.000000e+00,0
6,F,4.927540e+05,15,4.927540e+05,15,0.000000e+00,0
7,G,1.048811e+08,585,1.048811e+08,585,0.000000e+00,0
8,H,2.597230e+07,58,2.597230e+07,58,0.000000e+00,0


In [84]:
# Summary for the full/clean dataframe (recipients)

summary_original_recipient = (
    alldaf_allntee_df
    .groupby("RECIPIENT_NTEE_FIRST_DIGIT", as_index=False, dropna=False)
    .agg(
        Total_Cash_Grants_Original=("RcpntTbl_CshGrntAmt", "sum"),
        Count_Original=("RcpntTbl_CshGrntAmt", "count")
    )
)

summary_no_DAF2DAF_recipient = (
    alldaf_filtered_df
    .groupby("RECIPIENT_NTEE_FIRST_DIGIT", as_index=False, dropna=False)
    .agg(
        Total_Cash_Grants_No_DAF2DAF=("RcpntTbl_CshGrntAmt", "sum"),
        Count_No_DAF2DAF=("RcpntTbl_CshGrntAmt", "count")
    )
)

recipient_summary_combined = summary_original_recipient.merge(
    summary_no_DAF2DAF_recipient,
    on="RECIPIENT_NTEE_FIRST_DIGIT",
    how="outer"
)

recipient_summary_combined["DAF2DAF_Grants"] = (
    recipient_summary_combined["Total_Cash_Grants_Original"] - 
    recipient_summary_combined["Total_Cash_Grants_No_DAF2DAF"]
)
recipient_summary_combined["Count_DAF2DAF_Grants"] = (
    recipient_summary_combined["Count_Original"] - 
    recipient_summary_combined["Count_No_DAF2DAF"]
)

recipient_summary_combined = recipient_summary_combined.sort_values(
    by="RECIPIENT_NTEE_FIRST_DIGIT", na_position="first"
)

recipient_summary_combined.head(30)
# print(grantor_summary_combined["Total_Cash_Grants_Original"].sum())
# print(grantor_summary_combined["Total_Cash_Grants_No_DAF2DAF"].sum())

,RECIPIENT_NTEE_FIRST_DIGIT,Total_Cash_Grants_Original,Count_Original,Total_Cash_Grants_No_DAF2DAF,Count_No_DAF2DAF,DAF2DAF_Grants,Count_DAF2DAF_Grants
27,NaN,8.782648e+09,60514,8.782648e+09,60514,0.000000e+00,0
0,0,3.676492e+08,6328,3.673490e+08,6324,3.002130e+05,4
1,A,3.032121e+09,41036,3.031730e+09,41029,3.914000e+05,7
2,B,1.226698e+10,75493,1.222566e+10,75423,4.132790e+07,70
3,C,2.010396e+09,14953,2.010319e+09,14948,7.766300e+04,5
4,D,1.012778e+09,18757,1.012778e+09,18757,0.000000e+00,0
5,E,3.476777e+09,21252,3.476626e+09,21243,1.506080e+05,9
6,F,4.725781e+08,7162,4.725781e+08,7162,0.000000e+00,0
7,G,6.208659e+08,8791,6.208354e+08,8789,3.058600e+04,2
8,H,4.214540e+08,3558,4.214540e+08,3558,0.000000e+00,0


In [85]:
print(recipient_summary_combined["Total_Cash_Grants_Original"].sum())
print(recipient_summary_combined["Total_Cash_Grants_No_DAF2DAF"].sum())

62004522644.0
57017667108.0


### Output results

In [86]:
# Output the alldaf grants files

full_output_df = alldaf_allntee_df.copy()
full_output_df.to_csv(full_output_path, index=False)

nod2d_output_df = alldaf_filtered_df.copy()
nod2d_output_df.to_csv(nod2doutput_path, index=False)

In [87]:
# Output the summary tables

grantor_summary_df = grantor_summary_combined.copy()
grantor_summary_df.to_csv(grantor_summary_path, index=False)

recipient_summary_df = recipient_summary_combined.copy()
recipient_summary_df.to_csv(recipient_summary_path, index=False)
